## Supervised Learning  
#### Clasificación Predictiva en Salud y Reconocimiento de Dígitos  

<p><img src="https://preview.redd.it/rfgtsej8fhv71.jpg?auto=webp&s=99a5d000ff2baaac79a8d15c7135c3677f105159" width="1000"></p>

---

En este proyecto se desarrollan dos aplicaciones de Aprendizaje Supervisado:

1. Un modelo de clasificación para estimar la probabilidad de que un paciente presente riesgo de ataque al corazón.
2. Un modelo de reconocimiento de dígitos manuscritos utilizando el dataset MNIST.

El desarrollo contempla el flujo completo de trabajo: análisis exploratorio, preparación de datos, entrenamiento de modelos, evaluación y generación de predicciones sobre nuevas observaciones.

Cada etapa será documentada detalladamente dentro del cuaderno para asegurar trazabilidad y claridad metodológica.

## Exploratory Data Analysis

Trabajaremos con el conjunto de datos **Heart Attack**, cuyo objetivo es identificar qué variables clínicas influyen en la probabilidad de que un paciente sufra un evento cardiovascular.

Antes de construir un modelo predictivo, es fundamental comprender la estructura del dataset, el comportamiento de las variables y la relación entre ellas.

### Variables del dataset

- **age:** Edad del paciente  
- **sex:** Sexo del paciente  
- **cp:** Tipo de dolor torácico (0–3)  
- **trtbps:** Presión arterial en reposo (mm Hg)  
- **chol:** Nivel de colesterol (mg/dl)  
- **fbs:** Glucosa en ayunas > 120 mg/dl (1 = Sí, 0 = No)  
- **restecg:** Resultado electrocardiográfico en reposo  
- **thalachh:** Frecuencia cardíaca máxima alcanzada  
- **oldpeak:** Depresión del segmento ST  
- **slp:** Pendiente del segmento ST  
- **caa:** Número de vasos principales afectados  
- **thall:** Resultado de prueba de esfuerzo  
- **exng:** Angina inducida por ejercicio (1 = Sí, 0 = No)  
- **output:** Variable objetivo (1 = Riesgo, 0 = No riesgo)

El siguiente paso consiste en cargar el dataset y analizar su estructura.

In [ ]:
# Archivo Heart Attack.csv - Factores asociados al riesgo cardiovascular

import pandas as pd

# Ruta absoluta del proyecto
ruta_heart = r"D:\Documentos\Ebac\Finales\Tarea M25-CD –RobertScience\data26part2\Heart Attack.csv"

# Carga del dataset
df = pd.read_csv(ruta_heart)

# Visualización inicial
print("Dimensiones del dataset:", df.shape)
df.head()

In [ ]:
# ==========================================
# Inspección estructural del dataset
# ==========================================

print("\nInformación general:")
df.info()

print("\nResumen estadístico:")
df.describe()

In [ ]:
# ==========================================
# Verificación de valores nulos
# ==========================================

valores_nulos = df.isnull().sum()

print("Valores nulos por variable:\n")
print(valores_nulos)

print("\nTotal de valores nulos en el dataset:", valores_nulos.sum())

In [ ]:
# ==========================================
# Distribución de la variable objetivo
# ==========================================

print("Distribución absoluta del target:\n")
print(df["output"].value_counts())

print("\nDistribución porcentual:\n")
print(df["output"].value_counts(normalize=True) * 100)

In [ ]:
# ==========================================
# Matriz de correlación
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación")
plt.show()

In [ ]:
# ==========================================
# Correlación específica con la variable objetivo
# ==========================================

correlacion_output = df.corr()["output"].sort_values(ascending=False)

print("Correlación de variables con el riesgo cardiovascular:\n")
print(correlacion_output)

In [ ]:
# ==========================================
# Visualización de variables relevantes
# ==========================================

variables_clave = correlacion_output.index[1:6]

for var in variables_clave:
    plt.figure()
    sns.boxplot(x="output", y=var, data=df)
    plt.title(f"Distribución de {var} según riesgo cardiovascular")
    plt.show()

## k-Nearest Neighbors

Una vez comprendida la estructura del dataset y las relaciones entre variables, se procede a construir un modelo de clasificación utilizando el algoritmo **k-Nearest Neighbors (k-NN)**.

Este algoritmo clasifica una observación en función de la clase predominante entre sus *k* vecinos más cercanos en el espacio de características.

Dado que k-NN basa su funcionamiento en distancias, es necesario:

1. Separar los datos en conjuntos de entrenamiento y prueba.
2. Escalar las variables para evitar sesgos derivados de diferentes magnitudes.
3. Evaluar el desempeño del modelo con métricas cuantitativas.

El objetivo es estimar correctamente la variable `output`, que indica la presencia o ausencia de riesgo cardiovascular.

In [ ]:
# ==========================================
# Preparación de datos para modelado
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Variables independientes
X = df.drop("output", axis=1)

# Variable objetivo
y = df["output"]

# División entrenamiento / prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Dimensión X_train:", X_train.shape)
print("Dimensión X_test:", X_test.shape)

In [ ]:
# ==========================================
# Escalamiento de variables
# ==========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ==========================================
# Entrenamiento del modelo k-NN
# ==========================================

from sklearn.neighbors import KNeighborsClassifier

# Se define el modelo con 6 vecinos
knn = KNeighborsClassifier(n_neighbors=6)

# Entrenamiento
knn.fit(X_train_scaled, y_train)

print("Modelo entrenado correctamente.")

In [ ]:
# ==========================================
# Evaluación del modelo
# ==========================================

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predicciones sobre conjunto de prueba
y_pred = knn.predict(X_test_scaled)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy del modelo:", round(accuracy, 4))

# Matriz de confusión
print("\nMatriz de confusión:\n")
print(confusion_matrix(y_test, y_pred))

# Reporte de clasificación
print("\nReporte de clasificación:\n")
print(classification_report(y_test, y_pred))

## Predicción

Una vez entrenado y evaluado el modelo, es posible utilizarlo para estimar la clasificación de nuevas observaciones.

En un entorno real, esta etapa permitiría analizar nuevos registros clínicos para determinar si un paciente presenta riesgo cardiovascular, con base en sus variables fisiológicas.

A continuación se simula una nueva observación para demostrar el funcionamiento del modelo.

In [ ]:
# ==========================================
# Simulación de una nueva observación
# ==========================================

import numpy as np

# Se crea una observación simulada respetando el orden de las variables
X_new = np.array([[52, 1, 2, 130, 250, 0, 1, 150, 1.2, 2, 0, 2, 0]])

# Escalamiento utilizando el mismo scaler entrenado
X_new_scaled = scaler.transform(X_new)

# Predicción
y_new_pred = knn.predict(X_new_scaled)

print("Predicción del modelo (0 = No riesgo, 1 = Riesgo):", y_new_pred[0])

## Reconocimiento de dígitos

En esta sección se trabajará con el dataset MNIST, un conjunto de datos ampliamente utilizado para tareas de clasificación multiclase.

Cada observación representa una imagen de un dígito manuscrito (0–9) de 28x28 píxeles.  
Cada píxel está representado como una variable numérica, por lo que el problema se transforma en una clasificación supervisada con 10 clases posibles.

El objetivo es entrenar un modelo capaz de identificar correctamente el dígito representado en cada imagen.

In [ ]:
# ==========================================
# Carga del dataset MNIST
# ==========================================

ruta_mnist = r"D:\Documentos\Ebac\Finales\Tarea M25-CD –RobertScience\data26part2\MNIST.csv"

digits = pd.read_csv(ruta_mnist)

print("Dimensiones del dataset MNIST:", digits.shape)
digits.head()

In [ ]:
# ==========================================
# Identificación de variables de píxeles
# ==========================================

# Columnas que contienen información de los píxeles
cols = [col for col in digits.columns if "pixel" in col.lower()]

print("Número de variables de píxeles:", len(cols))

In [ ]:
# ==========================================
# Preparación de datos para clasificación
# ==========================================

X_digits = digits[cols]
y_digits = digits["label"]

print("Dimensión X:", X_digits.shape)
print("Dimensión y:", y_digits.shape)

In [ ]:
# ==========================================
# Visualización de una imagen
# ==========================================

import matplotlib.pyplot as plt

# Seleccionamos una observación
index = 0

imagen = X_digits.iloc[index].values.reshape(28, 28)

plt.imshow(imagen, cmap="gray")
plt.title(f"Dígito real: {y_digits.iloc[index]}")
plt.axis("off")
plt.show()

## Train/Test

En modelos de Aprendizaje Supervisado es fundamental dividir el conjunto de datos en dos subconjuntos:

- **Entrenamiento:** utilizado para ajustar el modelo.
- **Prueba:** utilizado para evaluar su desempeño sobre datos no vistos previamente.

Esta separación permite estimar la capacidad de generalización del modelo y cuantificar su desempeño mediante métricas objetivas.

En este caso, el objetivo es predecir correctamente el dígito representado en cada imagen. Para ello:

1. Se dividirán los datos utilizando `train_test_split`.
2. Se entrenará un clasificador k-Nearest Neighbors.
3. Se evaluará el modelo mediante la métrica `accuracy_score`.

In [ ]:
# ==========================================
# División del dataset en entrenamiento y prueba
# ==========================================

from sklearn.model_selection import train_test_split

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits,
    y_digits,
    test_size=0.2,
    random_state=42
)

print("Dimensión X_train:", X_train_d.shape)
print("Dimensión X_test:", X_test_d.shape)
print("Dimensión y_train:", y_train_d.shape)
print("Dimensión y_test:", y_test_d.shape)

In [ ]:
# ==========================================
# Escalamiento de los datos
# ==========================================

from sklearn.preprocessing import StandardScaler

scaler_d = StandardScaler()

X_train_d_scaled = scaler_d.fit_transform(X_train_d)
X_test_d_scaled = scaler_d.transform(X_test_d)

In [ ]:
# ==========================================
# Reducción de dimensionalidad (PCA)
# ==========================================

from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_d_scaled)
X_test_pca = pca.transform(X_test_d_scaled)

print("Dimensión original:", X_train_d_scaled.shape[1])
print("Dimensión reducida:", X_train_pca.shape[1])

In [ ]:
# ==========================================
# Entrenamiento del modelo k-NN sobre datos reducidos
# ==========================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

knn_pca = KNeighborsClassifier(n_neighbors=5)

knn_pca.fit(X_train_pca, y_train_d)

print("Modelo entrenado sobre espacio reducido correctamente.")

In [ ]:
# ==========================================
# Evaluación del modelo con PCA
# ==========================================

y_pred_pca = knn_pca.predict(X_test_pca)

accuracy_pca = accuracy_score(y_test_d, y_pred_pca)

print("Accuracy con PCA:", round(accuracy_pca, 4))

print("\nMatriz de confusión:\n")
print(confusion_matrix(y_test_d, y_pred_pca))

print("\nReporte de clasificación:\n")
print(classification_report(y_test_d, y_pred_pca))

In [ ]:
# ==========================================
# Carga y preparación de imagen externa
# ==========================================

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Ruta de la imagen PNG
ruta_imagen = r"D:\Documentos\Ebac\Finales\Tarea M25-CD –RobertScience\data26part2\2.png"

# Cargar imagen en escala de grises
imagen = Image.open(ruta_imagen).convert("L")

# Redimensionar a 28x28
imagen = imagen.resize((28, 28))

# Convertir a arreglo numpy
imagen_array = np.array(imagen)

# Visualizar
plt.imshow(imagen_array, cmap="gray")
plt.title("Imagen externa 28x28")
plt.axis("off")
plt.show()

print("Dimensiones:", imagen_array.shape)
print("Rango de valores:", imagen_array.min(), "a", imagen_array.max())

In [ ]:
# ==========================================
# Predicción imagen externa usando PCA
# ==========================================

import pandas as pd

# Convertir a float
imagen_array = imagen_array.astype(float)

# Asegurar coherencia de contraste
# MNIST usa dígitos claros sobre fondo oscuro
if imagen_array.mean() > 127:
    imagen_array = 255 - imagen_array

# Aplanar a vector 1x784
imagen_vector = imagen_array.reshape(1, -1)

# Convertir a DataFrame con mismas columnas
imagen_df = pd.DataFrame(imagen_vector, columns=X_train_d.columns)

# Escalar con el mismo scaler
imagen_vector_scaled = scaler_d.transform(imagen_df)

# Reducir dimensión con el PCA entrenado
imagen_vector_pca = pca.transform(imagen_vector_scaled)

# Predicción final
prediccion = knn_pca.predict(imagen_vector_pca)

print("El modelo predice que el número es:", prediccion[0])

In [ ]:
# ==========================================
# Regresión Logística (con PCA)
# ==========================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_reg = LogisticRegression(max_iter=5000, random_state=42)
log_reg.fit(X_train_pca, y_train_d)

y_pred_log = log_reg.predict(X_test_pca)

accuracy_log = accuracy_score(y_test_d, y_pred_log)

print("Accuracy Regresión Logística:", round(accuracy_log, 4))

In [ ]:
# ==========================================
# Evaluación detallada - Regresión Logística
# ==========================================

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Matriz de confusión
cm_log = confusion_matrix(y_test_d, y_pred_log)

plt.figure(figsize=(8,6))
sns.heatmap(cm_log, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusión - Regresión Logística")
plt.xlabel("Predicción")
plt.ylabel("Valor Real")
plt.show()

# Reporte de clasificación
print("\nReporte de Clasificación - Regresión Logística:\n")
print(classification_report(y_test_d, y_pred_log))

In [ ]:
# ==========================================
# Comparación de desempeño
# ==========================================

print("Accuracy k-NN con PCA:", round(accuracy_pca,4))
print("Accuracy Regresión Logística:", round(accuracy_log,4))

if accuracy_log > accuracy_pca:
    print("\nLa Regresión Logística presenta mejor desempeño.")
elif accuracy_log < accuracy_pca:
    print("\nk-NN presenta mejor desempeño.")
else:
    print("\nAmbos modelos presentan desempeño similar.")

# Conclusiones Estratégicas y Reflexión Final

## 1. Modelo de Clasificación – Riesgo Cardiovascular

El análisis exploratorio permitió identificar relaciones relevantes entre variables clínicas y la variable objetivo. Se observaron asociaciones significativas entre indicadores como tipo de dolor torácico, frecuencia cardíaca máxima alcanzada y número de vasos afectados con la probabilidad de riesgo cardiovascular.

La implementación del algoritmo k-Nearest Neighbors demostró un desempeño consistente, logrando una capacidad adecuada de discriminación entre pacientes con y sin riesgo.  

Desde una perspectiva técnica, el escalamiento de variables fue un paso crítico debido a la naturaleza basada en distancias del algoritmo. Sin esta etapa, el modelo hubiera presentado sesgos derivados de magnitudes heterogéneas.

Este modelo puede considerarse una herramienta de apoyo para sistemas de alerta temprana en entornos clínicos, donde la identificación oportuna de pacientes de alto riesgo puede contribuir a decisiones médicas más informadas.

---

## 2. Reconocimiento de Dígitos – MNIST

El dataset MNIST representa un problema de clasificación multiclase con alta dimensionalidad (784 variables).  

La aplicación de **Análisis de Componentes Principales (PCA)** permitió reducir la dimensionalidad conservando el 95% de la varianza explicada, lo que optimizó significativamente el desempeño computacional sin sacrificar precisión.

Se entrenaron dos modelos:

- k-Nearest Neighbors
- Regresión Logística

La comparación cuantitativa evidenció que ambos modelos presentan alta capacidad predictiva. La evaluación mediante matriz de confusión y reporte de clasificación permitió analizar el comportamiento por clase, identificando posibles confusiones entre dígitos visualmente similares.

La validación con una imagen externa dibujada manualmente confirmó la correcta implementación del pipeline completo:

1. Preprocesamiento
2. Escalamiento
3. Reducción de dimensionalidad
4. Clasificación

Esto demuestra capacidad de generalización más allá del conjunto de entrenamiento.

---

## 3. Análisis Comparativo de Modelos

La comparación entre k-NN y Regresión Logística permite extraer conclusiones relevantes:

- k-NN tiende a capturar patrones locales basados en proximidad.
- Regresión Logística modela fronteras de decisión globales de forma paramétrica.
- Ambos algoritmos pueden alcanzar desempeños competitivos en problemas bien estructurados.

La selección final de modelo en un entorno productivo dependería de factores como:

- Tiempo de inferencia
- Escalabilidad
- Interpretabilidad
- Requerimientos computacionales

---

## 4. Reflexión Técnica

Este proyecto integra de manera completa el ciclo de trabajo en Aprendizaje Supervisado:

- Comprensión del problema
- Análisis exploratorio
- Preparación y transformación de datos
- División entrenamiento/prueba
- Entrenamiento de modelos
- Evaluación con métricas adecuadas
- Validación en datos externos

Se refuerza la importancia de la correcta ingeniería de características y del preprocesamiento como factores determinantes en el desempeño de modelos predictivos.

Más allá del resultado numérico de accuracy, el verdadero valor radica en la capacidad de construir pipelines reproducibles, interpretables y generalizables.

---

## 5. Conclusión General

Los resultados obtenidos demuestran que los modelos de Aprendizaje Supervisado pueden aplicarse exitosamente tanto en contextos clínicos como en problemas de visión computacional.

La integración de análisis estadístico, reducción de dimensionalidad y evaluación comparativa permite transformar datos estructurados y no estructurados en herramientas predictivas robustas.

Este trabajo evidencia dominio en:

- Modelado de clasificación binaria y multiclase
- Técnicas de reducción de dimensionalidad
- Evaluación comparativa de algoritmos
- Implementación de pipelines completos de Machine Learning

El proyecto cumple con los objetivos planteados y demuestra la aplicabilidad práctica de los modelos desarrollados.

# Reflexión Personal sobre el Aprendizaje

Durante el desarrollo de este proyecto comprendí que el Aprendizaje Supervisado no se limita únicamente a entrenar un modelo y obtener un valor de accuracy. El verdadero trabajo comienza desde el análisis exploratorio, donde se interpreta la información y se identifican patrones relevantes.

En el caso del dataset de riesgo cardiovascular, entendí la importancia de analizar las variables antes de modelar. No todas las variables tienen el mismo impacto y la correlación permitió visualizar qué factores influyen más en la predicción.

También confirmé que el escalamiento es fundamental cuando se trabaja con algoritmos basados en distancias como k-NN. Sin una correcta normalización, el modelo puede verse afectado por diferencias en magnitudes.

En el problema de reconocimiento de dígitos, comprendí cómo la alta dimensionalidad puede afectar el desempeño y cómo técnicas como PCA ayudan a optimizar el modelo sin perder información relevante. Reducir dimensiones no significa perder calidad, sino trabajar de forma más eficiente.

Otro aprendizaje importante fue la comparación entre modelos. No existe un algoritmo universalmente mejor; cada modelo tiene ventajas dependiendo del problema. Evaluar y comparar es parte esencial del proceso.

Finalmente, este proyecto me permitió entender el flujo completo de un pipeline de Machine Learning: desde la exploración, preparación y transformación de datos, hasta la validación con datos nuevos. Esto refuerza la importancia de la metodología y no solo del resultado numérico.